In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

import joblib

# Data Load

In [2]:
df = pd.read_csv('../data/raw/startup_success_dataset.csv')

In [3]:
df.head()

,funding_rounds,founder_experience_years,team_size,market_size_billion,product_traction_users,burn_rate_million,revenue_million,investor_type,sector,founder_background,outcome
0,4,13,58,48.225483,594843,18.519211,1.483962e+06,tier2_vc,Health,academic,IPO
1,1,6,221,31.532647,393020,14.298149,8.620568e+05,tier2_vc,Fintech,first_time,Failure
2,3,5,247,4.969722,27636,20.447567,9.726169e+04,none,SaaS,first_time,Failure
3,3,14,229,3.084209,235376,8.177417,1.145785e+06,none,Ecommerce,ex_bigtech,Acquisition
4,1,17,235,13.818188,391765,4.879792,8.608949e+05,none,Health,first_time,Acquisition


# 4 - Feature Engineering

In [4]:
# Burn Efficiency — revenue generated per unit of burn
df['burn_efficiency'] = df['revenue_million'] / df['burn_rate_million']

# Revenue per Employee — team productivity metric
df['revenue_per_employee'] = df['revenue_million'] / df['team_size']

# Traction per Employee — product growth per team member
df['traction_per_employee'] = df['product_traction_users'] / df['team_size']

# Runway Risk — 1 if burn rate exceeds revenue, else 0
df['runway_risk'] = (df['burn_rate_million'] > df['revenue_million']).astype(int)

df.head()

,funding_rounds,founder_experience_years,team_size,market_size_billion,product_traction_users,burn_rate_million,revenue_million,investor_type,sector,founder_background,outcome,burn_efficiency,revenue_per_employee,traction_per_employee,runway_risk
0,4,13,58,48.225483,594843,18.519211,1.483962e+06,tier2_vc,Health,academic,IPO,80130.974239,25585.559419,10255.913793,0
1,1,6,221,31.532647,393020,14.298149,8.620568e+05,tier2_vc,Fintech,first_time,Failure,60291.496492,3900.709596,1778.371041,0
2,3,5,247,4.969722,27636,20.447567,9.726169e+04,none,SaaS,first_time,Failure,4756.638592,393.772008,111.886640,0
3,3,14,229,3.084209,235376,8.177417,1.145785e+06,none,Ecommerce,ex_bigtech,Acquisition,140115.809457,5003.429860,1027.842795,0
4,1,17,235,13.818188,391765,4.879792,8.608949e+05,none,Health,first_time,Acquisition,176420.429022,3663.382732,1667.085106,0


# 5 - Target Variable Encoding

In [5]:
df['outcome'] = df['outcome'].map({'Failure': 0, 'Acquisition': 1, 'IPO': 1})
df['outcome'].value_counts()

outcome
0    55610
1    44390
Name: count, dtype: int64

# 6. Train Test Split

In [6]:
X = df.drop('outcome', axis=1)
y = df['outcome']

# stratify=y — maintains same class ratio in both train and test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2, stratify=y)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}") 
print(f"y_test shape: {y_test.shape}") 

X_train shape: (80000, 14)
X_test shape: (20000, 14)
y_train shape: (80000,)
y_test shape: (20000,)


# 7 - Preprocessing Pipeline

In [7]:
cat_cols = ['investor_type', 'sector', 'founder_background']

numeric_cols = ['funding_rounds', 'founder_experience_years', 'team_size',
                'market_size_billion', 'product_traction_users',
                'burn_rate_million', 'revenue_million',
                'burn_efficiency', 'revenue_per_employee',
                'traction_per_employee', 'runway_risk']

# ColumnTransformer — applies different preprocessing to different column types
# fit_transform vs transform — fit only on train to prevent data leakage
# handle_unknown='ignore' — unseen categories in test set won't cause error
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numeric_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
])

# 8. Save Preprocessor & Raw Splits

In [8]:
# Save preprocessor
joblib.dump(preprocessor, '../models/preprocessor.joblib')

# Save processed data
X_train.to_csv('../data/processed/X_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

print("Preprocessor and raw splits saved successfully!") 

Preprocessor and raw splits saved successfully!
